# SHS generation + analysis

In [3]:
# Sequence & structure
PROTEIN_SEQ = "ACDEFGHIKLMNPQRSTVWYACDEFGHIKLMNPQRSTVWY"
PAIRS = "[[0, 3, 0.4], [5, 6, 1], [7, 12, 0.4], [13, 14, 1], [15, 16, 0.4], [17, 18, 1], [19, 20, 0.4]]"  # residue-index contact pairs
MUTATION_RATES = None

# MSA size / reproducibility
N = 1000
SEED = 42

# Mutation parameters
MUTATION_RATE_PAIRED = 0.3
MUTATION_RATE_UNPAIRED = 0.1
PAIR_MUTATION_APPROACH = "covariance"  # covariance | none

# insertions / deletions - paired
STEM_SINGLE_INSERTION_PROB  = 0.05
STEM_LONG_INSERTION_PROB    = 0.01
STEM_SINGLE_DELETION_PROB   = 0.1
STEM_PAIR_DELETION_PROB     = 0.2
STEM_MAX_INSERTION_FRACTION = 0.2

# insertions / deletions - unpaired
LOOP_SINGLE_INSERTION_PROB = 0.1
LOOP_SINGLE_DELETION_PROB  = 0.3
LOOP_LONG_INSERTION_PROB   = 0.1
LOOP_LONG_DELETION_PROB    = 0.05
LOOP_MAX_INSERTION_FRACTION = 0.2
LOOP_MAX_DELETION_FRACTION  = 0.1


## Setup

In [ ]:
import importlib
import sys
from pathlib import Path

%matplotlib inline

_roots = (Path.cwd(), *Path.cwd().parents)
REPO_ROOT = next((p for p in _roots if (p / "SHS-Generator-Protein" / "shs_generator.py").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the SyntheticEvolution repository root.")
SHS_DIR = (REPO_ROOT / "SHS-Generator-Protein").resolve()
ANALYZER_DIR = (REPO_ROOT / "analysis" / "SHS-Analyzer-Protein").resolve()
for module_dir in (SHS_DIR, ANALYZER_DIR):
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

import pair_map, generator_config, json_generator, shs_generator, shs_analyzer
importlib.reload(pair_map); importlib.reload(generator_config)
importlib.reload(json_generator); importlib.reload(shs_generator)
importlib.reload(shs_analyzer)

print("SHS-Generator dir:", SHS_DIR)
print("SHS-Analyzer dir:", ANALYZER_DIR)

ModuleNotFoundError: No module named 'pair_map'

## Generate the SHS

In [ ]:
structure_input = shs_generator.parse_json(PAIRS) if isinstance(PAIRS, str) else PAIRS
per_position_rates = shs_generator.parse_json(MUTATION_RATES) if isinstance(MUTATION_RATES, str) else MUTATION_RATES
pair_map = generator_config.build_pair_map(
    PROTEIN_SEQ, structure_input,
    mutation_rate_paired=MUTATION_RATE_PAIRED,
    mutation_rate_unpaired=MUTATION_RATE_UNPAIRED,
    mutation_rates=per_position_rates,
)

# Only global algorithm settings belong in MutationParameters.
parameters = generator_config.MutationParameters(
    N=N,
    pair_mutation_approach=PAIR_MUTATION_APPROACH,
    stem_single_insertion_prob=STEM_SINGLE_INSERTION_PROB,
    stem_long_insertion_prob=STEM_LONG_INSERTION_PROB,
    stem_single_deletion_prob=STEM_SINGLE_DELETION_PROB,
    stem_pair_deletion_prob=STEM_PAIR_DELETION_PROB,
    stem_max_insertion_fraction=STEM_MAX_INSERTION_FRACTION,
    loop_single_insertion_prob=LOOP_SINGLE_INSERTION_PROB,
    loop_single_deletion_prob=LOOP_SINGLE_DELETION_PROB,
    loop_long_insertion_prob=LOOP_LONG_INSERTION_PROB,
    loop_long_deletion_prob=LOOP_LONG_DELETION_PROB,
    loop_max_insertion_fraction=LOOP_MAX_INSERTION_FRACTION,
    loop_max_deletion_fraction=LOOP_MAX_DELETION_FRACTION,
)
generator = shs_generator.MsaGenerator(parameters, seed=SEED)
msa = generator.generate(PROTEIN_SEQ, pair_map)

# JSON construction is separate from generation. This notebook does not save it.
generated_json = json_generator.build_input_json(
    PROTEIN_SEQ, msa, name="notebook_protein", model_seeds=[1],
)
print("Generated", len(msa), "MSA rows for", len(PROTEIN_SEQ), "residues")


## Analyze & render

In [ ]:
import json
import numpy as np

data = shs_analyzer.MsaData.from_text(json.dumps(generated_json), pairs=pair_map)
feat = shs_analyzer.Features(data)

print(f"Query length: {len(data.seq)}  "
   f"|  MSA rows: {data.aligned.shape[0] + 1}  "
   f"|  covariance shape: {feat.covariance.shape}")

print(data.seq)
for row in data.raw[:10]:
    print("".join(row))


## Covariance per pair

In [ ]:
_ = shs_analyzer.plot_covariance_classification(data.seq, feat.covariance, pairs=data.pairs)


## Deletion and Insertion analysis

In [ ]:
_ = shs_analyzer.plot_deletions_and_insertion(feat.col_deletion_rate, feat.col_insertion_mean, pairs=data.pairs)


## Mutation-rate comparison

In [ ]:
# PairMap stores the effective rates, including scalar defaults and overrides.
input_rates = np.array([pair_map.mutation_rate(i) for i in range(len(data.seq))])
output_rates = feat.col_mut_rate
fig, metrics = shs_analyzer.plot_mutation_rate_comparison(input_rates, output_rates)
print(f"MAE: {metrics['mae']:.4f} | RMSE: {metrics['rmse']:.4f} | Pearson r: {metrics['pearson_r']:.4f}")


## Recovered parameters

In [ ]:
import pandas as pd

# Show full cell contents so the `detail` strings aren't truncated.
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

# Known generator input per recovered parameter name (the ground truth each
# estimator tries to reproduce). Extend this as you add estimators.
KNOWN_INPUTS = {
    "n_sequences":            N,
    "mutation_rate_unpaired": MUTATION_RATE_UNPAIRED,
    "mutation_rate_paired":   MUTATION_RATE_PAIRED,
}

report = shs_analyzer.analyze_data(data)


def _fmt_detail(detail):
    # Turn {'unpaired_cols': 14, ...} into a readable "unpaired_cols=14 · ..." line.
    return " · ".join(f"{k}={v}" for k, v in detail.items())


rows = []
for name, est in report.items():
    known = KNOWN_INPUTS.get(name)
    # est.value != est.value is True only for NaN -> no error there.
    error = abs(est.value - known) if (known is not None and est.value == est.value) else None
    rows.append({
        "parameter": name,
        "known":     known,
        "recovered": round(est.value, 4),
        "abs_error": round(error, 4) if error is not None else None,
        "detail":    _fmt_detail(est.detail),
    })

pd.DataFrame(rows).set_index("parameter")
